# Uncertainty-Aware Multi-Horizon LTE Handover and QoE Forecasting
### End-to-end training pipeline on real drive-test data

This notebook runs the whole study, in protocol order, on the `hoproj` library.
It does not re-implement anything: every cell calls the same code the CLI stages call,
so a number produced here and a number produced by `make all` are the same number.

| Section | Proposal step | Question it answers |
|---|---|---|
| 1 | setup | what is installed, what will be run |
| 2 | R0 | what can this capture actually measure? |
| 3 | R7–R8 | how do we turn one long log into drives, events and labels? |
| 4 | R9–R10 | what do the features and the class balance look like? |
| 5 | R12 | how is the data partitioned, and why that way? |
| 6 | R11–R12 | do sequence models beat snapshot models? (RQ1, RQ7) |
| 7 | R25 | how much performance does random row splitting invent? (RQ3) |
| 8 | R13 | do mobility and QoE inputs earn their place? (RQ5) |
| 9 | R14 | is the model's confidence trustworthy, and when should it abstain? (RQ4) |
| 10 | R15 | does any of it survive an unseen route? (RQ2, RQ6) |
| 11 | R16 | the consolidated report |

**A note on reading the results.** The positive class is rare — roughly 3% of samples at
the 1 s horizon. Accuracy is therefore meaningless and is never reported. AUPRC is the
primary metric and is always shown next to the prevalence it must beat.

---
## 1. Setup

In [1]:
import os, sys, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 9, 'axes.grid': True,
                     'grid.alpha': 0.3, 'figure.autolayout': True})

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
os.environ['HOPROJ_ROOT'] = str(ROOT)
print('project root:', ROOT)

project root: /home/claude/ho-pipeline


In [2]:
from hoproj.config import Config, deep_merge, load_config, resolve_paths
from hoproj.utils import environment_manifest, set_seed

env = environment_manifest()
print('python  :', env['python'])
for k, v in env['packages'].items():
    print(f'{k:<10}: {v}')

import torch
torch.set_num_threads(max(1, (os.cpu_count() or 2)))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('compute :', DEVICE, f'({os.cpu_count()} cores)')

python  : 3.11.15
numpy     : 2.4.4
pandas    : 3.0.2
scipy     : 1.17.1
sklearn   : 1.8.0
torch     : 2.14.0+cu130
lightgbm  : 4.7.0
pyarrow   : 25.0.1
compute : cpu (2 cores)


### The training budget

Everything below reads from one place. `BUDGET` is the only thing to change when moving
from a laptop to a GPU box — the protocol itself lives in `configs/base.yaml` and is not
touched here, which is the point of the R2 freeze.

In [3]:
BUDGET = {
    'train': {'epochs': int(os.environ.get('HO_EPOCHS', 25)),
              'early_stop_patience': 6,
              'batch_size': 256,
              'device': DEVICE},
    'uncertainty': {'ensemble_size': int(os.environ.get('HO_ENSEMBLE', 3))},
    'eval': {'bootstrap': {'n': int(os.environ.get('HO_BOOT', 300))}},
    'deploy': {'profile_repeats': 20, 'profile_batch_sizes': [1, 32, 256]},
}

BASE = load_config('base.yaml', adapter='curated_v1', overrides=BUDGET)
PATHS = resolve_paths(BASE, ROOT)
set_seed(BASE.get_path('project.seed'))

print('config fingerprint :', BASE.fingerprint)
print('horizons (s)       :', BASE.get_path('labels.horizons_s'))
print('window length (s)  :', BASE.get_path('windows.length_s'))
print('feature regime     :', BASE.get_path('features.regime'))
print('LOCKED route       :', BASE.get_path('splits.external_route'))
print('epochs             :', BUDGET['train']['epochs'])

config fingerprint : ead9c9d0bfad
horizons (s)       : [1.0, 2.0, 3.0, 5.0]
window length (s)  : 10.0
feature regime     : topology_agnostic
LOCKED route       : kuril_badda_rampura_malibagh
epochs             : 25


---
## 2. R0 — the feasibility gate

Before any modelling decision, one question: *what can this logging configuration actually
measure?* The gate reads a raw capture and returns a verdict per task. Running it on the raw
XCAL pilot export is more informative than running it on the curated file, because the pilot
is what the main campaign will produce.

In [4]:
from hoproj.pipeline import stage00_field_dictionary as S0

pilot_root = ROOT / 'pilot'
(pilot_root / 'data' / 'raw').mkdir(parents=True, exist_ok=True)
src_pilot = ROOT / 'data' / 'raw' / 'test 10 sept-M1.csv'
if src_pilot.exists():
    import shutil
    shutil.copy(src_pilot, pilot_root / 'data' / 'raw' / src_pilot.name)

    cfg_pilot = load_config('base.yaml', adapter='xcal_wide',
                            overrides={'data': {'sources': {'samples': src_pilot.name}}})
    paths_pilot = resolve_paths(cfg_pilot, pilot_root)
    field_dict = S0.run(cfg_pilot, paths_pilot)
    gate = pd.read_csv(paths_pilot['reports'] / 'tables' / 'xcal_feasibility_gate.csv')
    display(gate)
else:
    print('no raw XCAL export present; skipping the gate')
    field_dict, gate = None, None

16:07:11 | INFO    | hoproj.ingest          | reading samples from /home/claude/ho-pipeline/pilot/data/raw/test 10 sept-M1.csv


16:07:11 | INFO    | hoproj.ingest          | samples: 2756 rows, 2026-09-10 17:45:43 -> 2026-09-10 18:31:39


16:07:11 | INFO    | hoproj.report          | wrote report /home/claude/ho-pipeline/pilot/reports/00_field_dictionary.md


16:07:11 | INFO    | hoproj.stage00         | field dictionary written; verdicts: {'multi-horizon handover forecasting': 'supported', 'candidate-neighbour target ranking': 'NOT supported', 'QoE degradation prediction': 'degraded', 'QoE regression (RTT / loss)': 'NOT supported', 'mobility features': 'supported', 'signalling-confirmed handover / target cell': 'NOT supported', 'handover failure / RLF labels': 'NOT supported', "A3 rule baseline with the network's own parameters": 'NOT supported'}


,task,requires,field_status,verdict
0,multi-horizon handover forecasting,"serving_rsrp, serving_pci","available, available",supported
1,candidate-neighbour target ranking,"nbr1_rsrp, nbr1_id","absent, absent",NOT supported
2,QoE degradation prediction,dl_tp_kbps,sparse,degraded
3,QoE regression (RTT / loss),"rtt_ms, pkt_loss_pct","absent, absent",NOT supported
4,mobility features,"speed_kmh, lat, lon","available, available, available",supported
5,signalling-confirmed handover / target cell,"sig_handover_command, sig_target_pci","absent, absent",NOT supported
6,handover failure / RLF labels,sig_reestablishment,absent,NOT supported
7,A3 rule baseline with the network's own parame...,"sig_a3_offset, sig_time_to_trigger","absent, absent",NOT supported


In [5]:
if field_dict is not None:
    avail = field_dict.groupby(['group', 'availability']).size().unstack(fill_value=0)
    ax = avail.plot(kind='barh', stacked=True, figsize=(7, 3.2),
                    color={'available': '#2a9d8f', 'sparse': '#e9c46a', 'absent': '#e76f51'})
    ax.set_xlabel('number of fields'); ax.set_ylabel('')
    ax.set_title('What the raw XCAL pilot export actually contains')
    ax.legend(title='', loc='lower right', fontsize=8)
    plt.show()

**Read this before designing the campaign.** Any task marked *NOT supported* is not a
modelling problem — it is a logging-profile problem, and it has to be fixed in XCAL before
the main drives, not after.

---
## 3. R7–R8 — from raw log to labelled drives

Four things happen here, and the order matters:

1. **Ingest** — parse timestamps, map vendor column names onto canonical ones, detect
   logging sessions, put each session on a regular grid, flag every synthesised row.
2. **Segment** — cut each continuous session into individual traversals. This is the step
   that makes grouped evaluation possible at all; section 3.1 explains why.
3. **Quality control** — reject drives with clock gaps, missing GPS or implausible speed.
4. **Label** — multi-horizon handover targets, with masks, plus ping-pong, dwell, target
   cell and QoE degradation.

In [6]:
from hoproj.data import ingest, segment, qc
from hoproj.data import labels as L
from hoproj.data.features import build_features
from hoproj.data.splits import make_splits
from hoproj.utils import timed

samples, events = ingest.ingest(BASE, PATHS['raw'])
samples = segment.assign_routes(samples, events, BASE)
print(samples[['t', 'route_id', 'serving_rsrp', 'serving_pci', 'speed_kmh', 'dl_tp_kbps']].head())

16:07:12 | INFO    | hoproj.ingest          | reading samples from /home/claude/ho-pipeline/data/raw/DRIVETEST_LOGS_1_fixed.csv


16:07:13 | INFO    | hoproj.ingest          | samples: 81141 rows, 2026-09-06 09:00:00 -> 2026-09-08 14:19:57


16:07:13 | INFO    | hoproj.ingest          | regularised to 1.000s grid: 81141 rows (0.00% synthesised)


16:07:13 | INFO    | hoproj.ingest          | reading handover log from /home/claude/ho-pipeline/data/raw/HANDOVER_LOG_REGENERATED.csv


16:07:13 | INFO    | hoproj.ingest          | handover events: 6515 ({'pingpong': 2456, 'short_stay': 2411, 'normal': 1311, 'stable': 337})


                    t                  route_id  serving_rsrp  serving_pci  speed_kmh  dl_tp_kbps
0 2026-09-06 09:00:00  gulshan_banani_mohakhali         -80.0        203.0       10.2      2363.0
1 2026-09-06 09:00:01  gulshan_banani_mohakhali         -78.0        203.0       11.4     14087.0
2 2026-09-06 09:00:02  gulshan_banani_mohakhali         -78.0        203.0       13.5     23641.0
3 2026-09-06 09:00:03  gulshan_banani_mohakhali         -80.0        203.0       13.6      9206.0
4 2026-09-06 09:00:04  gulshan_banani_mohakhali         -76.0        203.0       16.7     31311.0


### 3.1 Why drives have to be reconstructed

The capture is a small number of *long* continuous sessions — one per corridor. Partitioning
by session would give one group per corridor: no grouped cross-validation, no trip-level
bootstrap, no honest confidence interval.

But a corridor session is not one journey. It is the same road driven up and down, repeatedly.
Projecting the GPS track onto the corridor's principal axis makes that visible as a sawtooth,
and each monotone leg of the sawtooth is one traversal — one drive.

In [7]:
from hoproj.utils import local_xy

demo = samples[samples['session_id'] == samples['session_id'].iloc[0]]
x, y = local_xy(demo['lat'].to_numpy(float), demo['lon'].to_numpy(float))
pts = np.column_stack([x, y]); pts = pts - pts.mean(axis=0)
axis = np.linalg.svd(pts, full_matrices=False)[2][0]
s_along = pts @ axis

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.plot(demo['lon'], demo['lat'], lw=0.4, color='#264653')
a1.set_title('GPS track of one corridor session'); a1.set_xlabel('lon'); a1.set_ylabel('lat')
a2.plot(demo['t_rel_s'] / 60, s_along / 1000, lw=0.7, color='#2a9d8f')
a2.set_title('the same session, projected onto the corridor axis')
a2.set_xlabel('minutes into session'); a2.set_ylabel('along-track position (km)')
plt.show()
print('each rising or falling leg above is one traversal = one drive')

each rising or falling leg above is one traversal = one drive


In [8]:
with timed('segment + QC'):
    samples = segment.segment_drives(samples, BASE)
    drives = segment.drive_table(samples)
    qc_tbl = qc.drive_quality(samples, drives, BASE)
    samples = qc.apply_qc(samples, qc_tbl)
    drives = drives[drives['drive_id'].isin(set(samples['drive_id']))].reset_index(drop=True)

print(f'{len(drives)} drives survive QC')
display(drives.groupby(['route_id', 'direction'])
              .agg(drives=('drive_id', 'nunique'),
                   median_min=('duration_s', lambda s: round(s.median() / 60, 1)),
                   median_km=('distance_m', lambda s: round(s.median() / 1000, 2)),
                   mean_kmh=('mean_speed_kmh', lambda s: round(s.mean(), 1))))

16:07:13 | INFO    | hoproj                 | START segment + QC


16:07:14 | INFO    | hoproj.segment         | segmented 81 drives over 3 routes (median 880s, median 8.53 km)


16:07:14 | INFO    | hoproj.qc              | QC: 81/81 drives pass ({})


16:07:14 | INFO    | hoproj.qc              | dropped 0 samples from rejected drives


16:07:14 | INFO    | hoproj                 | DONE  segment + QC (1.31s)


81 drives survive QC


drives  median_min  median_km  mean_kmh
route_id                      direction                                         
farmgate_sciencelab_newmarket fwd             6        37.7      12.31      14.0
                              rev             7        22.8       7.95      14.0
gulshan_banani_mohakhali      fwd            17        17.3       9.61      30.4
                              rev            16        16.8       9.00      29.3
kuril_badda_rampura_malibagh  fwd            18        12.1       8.42      42.1
                              rev            17        10.3       8.17      47.2

In [9]:
rejected = qc_tbl.loc[~qc_tbl['qc_pass']]
if len(rejected):
    display(rejected[['drive_id', 'qc_reject_reason', 'duration_s', 'missing_gps_frac', 'max_dt_s']])
else:
    print('no drive was rejected by quality control')

no drive was rejected by quality control


### 3.2 Labels

`y_ho_h{H} = 1` when a handover falls in the half-open interval `(t, t+H]`. Three masking
rules keep the target honest:

- samples within `post_event_blank_s` of a handover are **masked out** — the radio state is
  still settling and the next event is trivially predictable there;
- samples whose horizon runs past the end of their drive are **masked**, not labelled 0;
- a horizon shorter than one sample period is **dropped with a warning**, because on a
  regular grid it cannot carry a single positive label.

In [10]:
ho = L.handover_events(samples, events, BASE)
ho = ho[ho['drive_id'].isin(set(samples['drive_id']))].reset_index(drop=True)

splits = make_splits(drives, BASE)
train_drives = set(splits['grouped_drive'].train)
qoe_th = L.fit_qoe_thresholds(samples[samples['drive_id'].isin(train_drives)], BASE)
labels = L.build_labels(samples, ho, BASE, qoe_th)

print('QoE degradation thresholds, fitted on TRAINING drives only:')
for k, v in qoe_th.items():
    print(f'  {k:<22} {v}')

16:07:14 | INFO    | hoproj.labels          | handover events: 6515 (37.7% ping-pong) from handover_log


16:07:15 | INFO    | hoproj.splits          | split grouped_drive   train=25 val=7 calib=7 test=7 drives


16:07:15 | INFO    | hoproj.splits          | split external_route  train=25 val=7 calib=7 test=35 drives


16:07:15 | INFO    | hoproj.splits          | split random_row      train=46 val=46 calib=46 test=46 drives


16:07:15 | INFO    | hoproj.labels          | QoE thresholds (train-only): {'dl_tp_low': 2191.0, 'rtt_high': 99.1, 'loss_high': 2.38, 'interruption_dl_kbps': 50.0, 'min_rules': 1}


16:07:15 | INFO    | hoproj.labels          | HO label prevalence: 1.0s=6.06% (n=69399), 2.0s=11.07% (n=69332), 3.0s=15.31% (n=69265), 5.0s=22.26% (n=69131)


QoE degradation thresholds, fitted on TRAINING drives only:
  dl_tp_low              2191.0
  rtt_high               99.1
  loss_high              2.38
  interruption_dl_kbps   50.0
  min_rules              1


In [11]:
tags = L.horizon_tags(BASE)
horizons = L.usable_horizons(BASE)
prev = pd.DataFrame([{
    'horizon_s': h,
    'valid_samples': int(labels[f'm_ho_{t}'].sum()),
    'masked_out': int((labels[f'm_ho_{t}'] == 0).sum()),
    'HO_positive_rate': labels.loc[labels[f'm_ho_{t}'] == 1, f'y_ho_{t}'].mean(),
    'QoE_positive_rate': labels.loc[labels[f'm_ho_{t}'] == 1, f'y_qoe_{t}'].mean()
                         if f'y_qoe_{t}' in labels else np.nan,
} for h, t in zip(horizons, tags)])
display(prev.style.format({'HO_positive_rate': '{:.3%}', 'QoE_positive_rate': '{:.2%}'}))

,horizon_s,valid_samples,masked_out,HO_positive_rate,QoE_positive_rate
0,1.000000,69399,11742,6.059%,23.06%
1,2.000000,69332,11809,11.068%,38.71%
2,3.000000,69265,11876,15.306%,49.50%
3,5.000000,69131,12010,22.265%,62.37%


In [12]:
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.bar(prev['horizon_s'], prev['HO_positive_rate'] * 100, width=0.45, color='#264653',
       label='handover within horizon')
if prev['QoE_positive_rate'].notna().any():
    ax.bar(prev['horizon_s'] + 0.45, prev['QoE_positive_rate'] * 100, width=0.45,
           color='#e9c46a', label='QoE degradation within horizon')
ax.set_xlabel('forecast horizon (s)'); ax.set_ylabel('positive rate (%)')
ax.set_title('Class balance — this is the floor every AUPRC must beat')
ax.legend(fontsize=8)
plt.show()

---
## 4. R9–R10 — features and dataset characterisation

Every feature is causal: rolling statistics look backwards only, derivatives are backward
differences, and nothing is computed over a whole drive. The blocks are RF, mobility,
cell history and QoE; absolute position, route identity and cell identity live in a separate
`context` block that the *topology-agnostic* regime excludes entirely.

In [13]:
with timed('features'):
    feats = build_features(samples, labels, BASE)

key = ['drive_id', 't']
labels = feats[key].merge(labels, on=key, how='left', validate='one_to_one').reset_index(drop=True)

blocks = pd.Series(feats.attrs['feature_block']).value_counts().rename('features')
display(blocks.to_frame())
print(f"total: {len(feats.attrs['feature_names'])} features")

16:07:15 | INFO    | hoproj                 | START features


16:07:22 | INFO    | hoproj.features        | built 163 features ({'context': 16, 'history': 7, 'mask': 9, 'mobility': 17, 'qoe': 31, 'rf': 83}) under regime=topology_agnostic


16:07:22 | INFO    | hoproj                 | DONE  features (7.22s)


,features
rf,83
qoe,31
mobility,17
context,16
mask,9
history,7


total: 163 features


In [14]:
from hoproj.data.characterise import dataset_summary, cell_overlap, shift_report

summary = dataset_summary(samples, drives, ho, labels, BASE)
per_route = pd.DataFrame(summary['per_route'])
display(per_route.round(2))
print(f"{summary['n_handovers']} handovers | {summary['pingpong_frac']:.1%} ping-pong | "
      f"{summary['total_hours']:.1f} h | {summary['total_km']:.0f} km | "
      f"{summary['unique_serving_cells']} serving cells")

,route_id,drives,samples,hours,km,mean_speed,handovers,pingpong,ho_per_km
0,farmgate_sciencelab_newmarket,13,23441,6.51,126.76,14.04,1823,627,14.38
1,gulshan_banani_mohakhali,33,34302,9.52,304.11,29.90,2351,939,7.73
2,kuril_badda_rampura_malibagh,35,23398,6.49,293.80,44.55,2341,890,7.97


6515 handovers | 37.7% ping-pong | 22.5 h | 725 km | 46 serving cells


### 4.1 Persisting the processed tables

Sections 10 and 11 call the CLI stages, and those read from disk rather than from this
kernel's memory. Writing the tables now means the notebook and `make all` operate on
byte-identical inputs — which is the whole point of not re-implementing anything here.

In [15]:
from hoproj.utils import write_parquet, write_json

write_parquet(samples, PATHS['interim'] / 'samples.parquet')
write_parquet(drives, PATHS['processed'] / 'drives.parquet')
write_parquet(qc_tbl, PATHS['processed'] / 'qc_report.parquet')
write_parquet(ho, PATHS['processed'] / 'handovers.parquet')
write_parquet(labels, PATHS['processed'] / 'labels.parquet')
write_parquet(feats, PATHS['processed'] / 'features.parquet')
write_json({'feature_block': feats.attrs['feature_block'],
            'feature_names': feats.attrs['feature_names']},
           PATHS['processed'] / 'feature_meta.json')
write_json({k: v.as_dict() for k, v in splits.items()}, PATHS['processed'] / 'splits.json')
write_json({'summary': summary, 'qoe_thresholds': qoe_th,
            'config_fingerprint': BASE.fingerprint, 'environment': env},
           PATHS['processed'] / 'dataset_summary.json')
BASE.dump(PATHS['processed'] / 'frozen_config.yaml')
print('processed tables written to', PATHS['processed'])

processed tables written to /home/claude/ho-pipeline/data/processed


### 4.2 What an approaching handover looks like

The single most informative variable is not the serving RSRP but the *gap* between the
serving cell and the strongest candidate neighbour. Averaged over every real handover, the
gap collapses in the seconds before the event — which is exactly the signal a predictor has
to pick up, and exactly what the A3 rule uses.

In [16]:
gap_col = 'gap_serving_best_nbr' if 'gap_serving_best_nbr' in feats else 'gap_serving_nbr1'
win = 15
stack = []
idx_by_drive = feats.groupby('drive_id').indices
for d, grp in ho.groupby('drive_id'):
    pos = idx_by_drive.get(d)
    if pos is None:
        continue
    tt = feats['t'].to_numpy('datetime64[ns]')[pos]
    for e in grp['t'].to_numpy('datetime64[ns]'):
        j = int(np.searchsorted(tt, e))
        if win <= j < len(pos) - 3:
            stack.append([feats[gap_col].to_numpy()[pos[j - win:j + 3]],
                          feats['serving_rsrp'].to_numpy()[pos[j - win:j + 3]]])
stack = np.array(stack, dtype=float)
lag = np.arange(-win, 3)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, k, lbl in ((a1, 0, f'{gap_col} (dB)'), (a2, 1, 'serving RSRP (dBm)')):
    m = np.nanmean(stack[:, k, :], axis=0)
    q1, q3 = np.nanpercentile(stack[:, k, :], [25, 75], axis=0)
    ax.fill_between(lag, q1, q3, alpha=0.25, color='#2a9d8f')
    ax.plot(lag, m, color='#264653', lw=1.8)
    ax.axvline(0, color='#e76f51', ls='--', lw=1)
    ax.set_xlabel('seconds relative to handover'); ax.set_ylabel(lbl)
a1.set_title(f'serving-to-best-neighbour gap ({len(stack)} events)')
a2.set_title('serving RSRP')
plt.show()

---
## 5. R12 — partitioning

Four disjoint sets of **whole drives**, stratified by route and direction:

| part | used for | never used for |
|---|---|---|
| `train` | fitting parameters, scaler, imputer, QoE thresholds | anything reported |
| `val` | early stopping, operating thresholds | calibration, reporting |
| `calib` | temperature scaling, conformal, abstention thresholds | training, model choice |
| `test` | reporting, grouped-drive condition | any decision |

and one route held entirely outside all four, locked until section 10.

In [17]:
rows = []
for name, sp in splits.items():
    rows.append({'condition': name, 'train': len(sp.train), 'val': len(sp.val),
                 'calib': len(sp.calib), 'test': len(sp.test), 'note': sp.note})
display(pd.DataFrame(rows))

g = splits['grouped_drive']
parts = {'train': set(g.train), 'val': set(g.val), 'calib': set(g.calib), 'test': set(g.test)}
overlaps = {f'{a}&{b}': len(parts[a] & parts[b])
            for i, a in enumerate(parts) for b in list(parts)[i + 1:]}
assert max(overlaps.values()) == 0, overlaps
print('development partitions share no drive:', overlaps)
ext = set(splits['external_route'].test)
assert not ext & set(g.train) | ext & set(g.val) | ext & set(g.calib)
print(f'external route holds {len(ext)} drives, disjoint from every development part')

,condition,train,val,calib,test,note
0,grouped_drive,25,7,7,7,Condition B: complete-drive separation inside ...
1,external_route,25,7,7,35,Condition C: locked external route 'kuril_badd...
2,random_row,46,46,46,46,Condition A: rows are shuffled at sample level...


development partitions share no drive: {'train&val': 0, 'train&calib': 0, 'train&test': 0, 'val&calib': 0, 'val&test': 0, 'calib&test': 0}
external route holds 35 drives, disjoint from every development part


In [18]:
lab = drives.assign(part=np.select(
    [drives['drive_id'].isin(parts['train']), drives['drive_id'].isin(parts['val']),
     drives['drive_id'].isin(parts['calib']), drives['drive_id'].isin(parts['test']),
     drives['drive_id'].isin(ext)],
    ['train', 'val', 'calib', 'test', 'EXTERNAL (locked)'], default='unused'))
pivot = lab.pivot_table(index='route_id', columns='part', values='drive_id',
                        aggfunc='count', fill_value=0)
order = [c for c in ['train', 'val', 'calib', 'test', 'EXTERNAL (locked)'] if c in pivot]
ax = pivot[order].plot(kind='barh', stacked=True, figsize=(7.5, 2.8),
                       color=['#264653', '#2a9d8f', '#e9c46a', '#f4a261', '#e76f51'])
ax.set_xlabel('drives'); ax.set_ylabel('')
ax.set_title('Drive allocation — the locked route contributes to nothing before section 10')
ax.legend(fontsize=8, ncol=3)
plt.show()

---
## 6. R11–R12 — the model comparison (RQ1, RQ7)

Five models, identical data, identical splits, identical metrics:

- **rule** — an A3-inspired threshold on the serving-to-neighbour gap and its trend. Not fitted.
- **logreg** — a linear snapshot classifier.
- **lgbm** — a strong gradient-boosted snapshot baseline.
- **gru / tcn / transformer** — sequence models over a 10 s window.

All of them are assembled once and share the same feature matrix, so any difference is the
model and nothing else.

In [19]:
from hoproj.pipeline.assemble import assemble
from hoproj.pipeline.trainer import train_and_evaluate

MODELS = ['rule', 'logreg', 'lgbm', 'gru', 'tcn', 'transformer']

cfg_main = Config(deep_merge(BASE, {}))
A = assemble(feats, labels, cfg_main, splits['grouped_drive'], PATHS['interim'] / 'wcache')
print(f'{A.n_features} features, windows: '
      + ', '.join(f'{p}={len(A.windows[p])}' for p in ('train', 'val', 'calib', 'test')))

16:07:27 | INFO    | hoproj.transforms      | dropping 9 degenerate features (train-only decision): ['mask_serving_rsrp', 'mask_serving_rsrq', 'mask_serving_sinr', 'mask_dl_tp_kbps', 'mask_rtt_ms', 'mask_pkt_loss_pct', 'mask_nbr1_rsrp', 'mask_nbr2_rsrp'] ...


16:07:27 | INFO    | hoproj.transforms      | fitted robust transform on 29027 train rows x 138 columns


16:07:28 | INFO    | hoproj.windows         | window index: 80412 windows of length 10 (stride 1) over 81 drives


16:07:28 | INFO    | hoproj.windows         | materialised windows: shape=(28802, 10, 138) (151.6 MB)


16:07:28 | INFO    | hoproj.assemble        | part=train  rows= 29027 windows= 28802 drives= 25


...

16:07:29 | INFO    | hoproj.windows         | materialised windows: shape=(10030, 10, 138) (52.8 MB)


16:07:29 | INFO    | hoproj.assemble        | part=calib  rows= 10093 windows= 10030 drives=  7


16:07:29 | INFO    | hoproj.windows         | materialised windows: shape=(9599, 10, 138) (50.5 MB)


16:07:29 | INFO    | hoproj.assemble        | part=test   rows=  9662 windows=  9599 drives=  7


138 features, windows: train=28802, val=8898, calib=10030, test=9599


In [20]:
results, event_rows, runinfo, preds = [], [], {}, {}
for name in MODELS:
    cfg_m = load_config('base.yaml', adapter='curated_v1', model=name if name != 'rule' else 'rule',
                        overrides=BUDGET)
    t0 = time.perf_counter()
    res = train_and_evaluate(cfg_m, feats, labels, ho, splits['grouped_drive'], name,
                             feature_set='rf_mob_hist_qoe', assembled=A)
    runinfo[name] = {'seconds': round(time.perf_counter() - t0, 1), **(res.footprint or {})}
    preds[name] = res.predictions['test']
    results.append(res.horizon_table)
    if len(res.event_table):
        event_rows.append(res.event_table)
    print(f"{name:<12} done in {runinfo[name]['seconds']:>6.1f}s")

main_results = pd.concat(results, ignore_index=True)
main_events = pd.concat(event_rows, ignore_index=True) if event_rows else pd.DataFrame()

rule         done in    3.0s


logreg       done in   30.9s


lgbm         done in   56.6s


16:09:00 | INFO    | hoproj.wrapper         | gru pos_weight per horizon: [16.209999084472656, 8.369999885559082, 5.739999771118164, 3.619999885559082]


16:09:21 | INFO    | hoproj.wrapper         | gru ep001 train=1.1246 val=1.0131 metric=0.2619 (best 0.2619 @ep1)


...

16:12:34 | INFO    | hoproj.wrapper         | transformer ep005 train=0.8571 val=0.8701 metric=0.3329 (best 0.3329 @ep5)


16:13:18 | INFO    | hoproj.wrapper         | transformer ep010 train=0.6711 val=1.0498 metric=0.2998 (best 0.3357 @ep6)


16:13:36 | INFO    | hoproj.wrapper         | transformer ep012 train=0.5955 val=1.4660 metric=0.2774 (best 0.3357 @ep6)


16:13:36 | INFO    | hoproj.wrapper         | transformer early stop at epoch 12


transformer  done in  112.3s


In [21]:
view = ['model', 'horizon_s', 'positive_rate', 'auprc', 'auprc_ci_low', 'auprc_ci_high',
        'auprc_lift', 'auroc', 'recall_at_fpr0.05', 'brier', 'ece']
display(main_results[[c for c in view if c in main_results]]
        .round(4).set_index(['model', 'horizon_s']))

positive_rate   auprc  auprc_ci_low  auprc_ci_high  auprc_lift   auroc  recall_at_fpr0.05   brier     ece
model       horizon_s                                                                                                           
rule        1.0               0.0570  0.1256        0.1138         0.1387      2.2027  0.7113             0.1492  0.6306  0.7431
            2.0               0.1052  0.2101        0.1907         0.2268      1.9976  0.7069             0.1448  0.5917  0.6952
            3.0               0.1466  0.2830        0.2557         0.3068      1.9300  0.7024             0.1547  0.5600  0.6540
            5.0               0.2156  0.3919        0.3562         0.4225      1.8178  0.7026             0.1628  0.5083  0.5856
logreg      1.0               0.0570  0.1730        0.1492         0.2101      3.0330  0.8025             0.2248  0.1791  0.2728
            2.0               0.1052  0.2734        0.2400         0.3332      2.6003  0.7974             0.2007  0.1831  0.2473
            3.0               0.1466  0.3580        0.3164         0.4311      2.4414  0.7984             0.2087  0.1836  0.2219
            5.0               0.2156  0.4695        0.4282         0.5335      2.1779  0.7979             0.2074  0.1835  0.1765
lgbm        1.0               0.0570  0.1910        0.1710         0.2215      3.3497  0.8159             0.2479  0.0542  0.0371
            2.0               0.1052  0.3055        0.2713         0.3547      2.9049  0.8163             0.2246  0.0952  0.0643
            3.0               0.1466  0.3870        0.3486         0.4313      2.6388  0.8165             0.2390  0.1195  0.0751
            5.0               0.2156  0.4944        0.4502         0.5370      2.2930  0.8160             0.2419  0.1496  0.0882
gru         1.0               0.0570  0.1869        0.1534         0.2328      3.2781  0.8009             0.2416  0.1501  0.2190
            2.0               0.1052  0.2770        0.2458         0.3213      2.6343  0.7949             0.2041  0.1624  0.1960
            3.0               0.1466  0.3554        0.3203         0.4029      2.4235  0.7964             0.2062  0.1639  0.1611
            5.0               0.2156  0.4674        0.4205         0.5226      2.1679  0.7981             0.2062  0.1700  0.1252
tcn         1.0               0.0570  0.1879        0.1607         0.2292      3.2956  0.8033             0.2437  0.1906  0.2490
            2.0               0.1052  0.2919        0.2597         0.3366      2.7763  0.8024             0.2258  0.1909  0.2230
            3.0               0.1466  0.3643        0.3339         0.4016      2.4844  0.8031             0.2177  0.1920  0.2050
            5.0               0.2156  0.4765        0.4324         0.5195      2.2102  0.8040             0.2196  0.1896  0.1651
transformer 1.0               0.0570  0.1770        0.1497         0.2149      3.1031  0.8038             0.2521  0.1448  0.2071
            2.0               0.1052  0.2688        0.2351         0.3198      2.5564  0.7968             0.2189  0.1507  0.1783
            3.0               0.1466  0.3490        0.3110         0.4058      2.3797  0.7967             0.2119  0.1572  0.1553
            5.0               0.2156  0.4605        0.4185         0.5215      2.1361  0.7965             0.2101  0.1662  0.1205

### 6.1 Performance against horizon

The dashed line is the prevalence floor — the AUPRC a coin flip weighted to the base rate
would get. Distance above that line, not the absolute value, is the result.

In [22]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 3.8))
colors = dict(zip(MODELS, ['#8d99ae', '#457b9d', '#e9c46a', '#264653', '#2a9d8f', '#e76f51']))
for m, grp in main_results.groupby('model'):
    grp = grp.sort_values('horizon_s')
    a1.plot(grp['horizon_s'], grp['auprc'], marker='o', label=m, color=colors.get(m))
    a1.fill_between(grp['horizon_s'], grp.get('auprc_ci_low', grp['auprc']),
                    grp.get('auprc_ci_high', grp['auprc']), alpha=0.12, color=colors.get(m))
    a2.plot(grp['horizon_s'], grp['auprc_lift'], marker='o', label=m, color=colors.get(m))
base = main_results.groupby('horizon_s')['positive_rate'].mean().sort_index()
a1.plot(base.index, base.values, 'k--', lw=1, label='prevalence floor')
a1.set_xlabel('forecast horizon (s)'); a1.set_ylabel('AUPRC')
a1.set_title('AUPRC with drive-level 95% CI'); a1.legend(fontsize=7, ncol=2)
a2.set_xlabel('forecast horizon (s)'); a2.set_ylabel('AUPRC / prevalence')
a2.set_title('lift over the prevalence floor')
plt.show()

### 6.2 The metric that actually matters operationally

A mobility controller does not consume AUPRC. It consumes *warnings*, and it pays for every
false one. Event-level evaluation counts each physical handover once, asks whether a warning
arrived inside the horizon, and pairs the detection rate with the false-alarm rate at the
same operating point — a threshold fixed on the validation drives at 5% FPR.

In [23]:
if len(main_events):
    ev = ['model', 'horizon_s', 'n_events', 'event_detection_rate', 'median_lead_time_s',
          'false_alarms_per_hour', 'false_alarms_per_km', 'threshold']
    display(main_events[[c for c in ev if c in main_events]].round(3)
            .set_index(['model', 'horizon_s']))

n_events  event_detection_rate  median_lead_time_s  false_alarms_per_hour  false_alarms_per_km  threshold
model       horizon_s                                                                                                           
rule        1.0             682                 0.092                 1.0                 28.926                1.106      1.000
            2.0             682                 0.125                 2.0                 28.571                1.093      1.000
            3.0             682                 0.152                 3.0                 26.341                1.008      1.000
            5.0             682                 0.186                 5.0                 21.103                0.807      1.000
logreg      1.0             682                 0.160                 1.0                 73.630                2.815      0.831
            2.0             682                 0.226                 2.0                 65.789                2.516      0.829
            3.0             682                 0.308                 2.0                 58.325                2.231      0.828
            5.0             682                 0.406                 4.0                 50.120                1.918      0.821
lgbm        1.0             682                 0.169                 1.0                 88.657                3.390      0.265
            2.0             682                 0.254                 2.0                 77.444                2.962      0.566
            3.0             682                 0.304                 2.0                 65.475                2.505      0.694
            5.0             682                 0.418                 4.0                 47.106                1.802      0.798
gru         1.0             682                 0.167                 1.0                 63.487                2.427      0.781
            2.0             682                 0.233                 2.0                 61.654                2.358      0.803
            3.0             682                 0.290                 2.0                 53.434                2.044      0.798
            5.0             682                 0.406                 4.0                 49.367                1.889      0.797
tcn         1.0             682                 0.161                 1.0                 74.382                2.844      0.863
            2.0             682                 0.258                 2.0                 70.677                2.703      0.863
            3.0             682                 0.331                 2.0                 68.485                2.620      0.866
            5.0             682                 0.468                 4.0                 61.803                2.365      0.868
transformer 1.0             682                 0.182                 1.0                 72.879                2.787      0.799
            2.0             682                 0.248                 2.0                 69.925                2.674      0.791
            3.0             682                 0.323                 2.0                 64.722                2.476      0.796
            5.0             682                 0.446                 4.0                 53.889                2.062      0.796

In [24]:
if len(main_events):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 3.8))
    for m, grp in main_events.groupby('model'):
        grp = grp.sort_values('horizon_s')
        a1.plot(grp['horizon_s'], grp['event_detection_rate'], marker='o', label=m,
                color=colors.get(m))
        a2.scatter(grp['false_alarms_per_hour'], grp['event_detection_rate'],
                   label=m, color=colors.get(m), s=34)
        for _, r in grp.iterrows():
            a2.annotate(f"{r['horizon_s']:g}s", (r['false_alarms_per_hour'],
                        r['event_detection_rate']), fontsize=6,
                        xytext=(3, 3), textcoords='offset points')
    a1.set_xlabel('forecast horizon (s)'); a1.set_ylabel('fraction of real handovers warned')
    a1.set_title('event detection rate'); a1.legend(fontsize=7, ncol=2)
    a2.set_xlabel('false alarms per hour'); a2.set_ylabel('event detection rate')
    a2.set_title('the operating trade-off — up and to the LEFT is better')
    a2.legend(fontsize=7, ncol=2)
    plt.show()

### 6.3 Is the difference real?

Two models trained on the same 42 drives will differ by chance. The comparison below
resamples **whole drives**, not rows — rows inside a drive are strongly dependent, and a
row-level bootstrap would produce confidence intervals several times too narrow.

In [25]:
from sklearn.metrics import average_precision_score
from hoproj.eval import stats as ST
from hoproj.pipeline.assemble import window_meta

pos_t = A.windows['test'].end_pos
Yt, Mt = A.Y_ho[pos_t], A.M_ho[pos_t]
groups = window_meta(A, 'test')['drive_id'].to_numpy()

j = min(range(len(horizons)), key=lambda i: abs(horizons[i] - 2.0))
sel = Mt[:, j].astype(bool)
auprc = lambda y, p: average_precision_score(y, p) if len(np.unique(y)) > 1 else np.nan

best = max(MODELS, key=lambda m: auprc(Yt[sel, j], preds[m][sel, j]))
comp = []
for m in MODELS:
    if m == best:
        continue
    perm = ST.paired_permutation(auprc, Yt[sel, j], preds[best][sel, j], preds[m][sel, j],
                                 groups[sel], n=400)
    wil = ST.wilcoxon_per_drive(auprc, Yt[sel, j], preds[best][sel, j], preds[m][sel, j], groups[sel])
    comp.append({'comparison': f'{best} vs {m}', 'delta_auprc': perm['observed_difference'],
                 'permutation_p': perm['p_value'], 'wilcoxon_p': wil.get('p_value'),
                 'n_drives': wil.get('n_drives')})
comp = pd.DataFrame(comp)
holm = ST.holm_correction(dict(zip(comp['comparison'], comp['permutation_p'])))
display(comp.merge(holm, on='comparison').round(4))
print(f'best model at the {horizons[j]:g}s horizon: {best}')

,comparison,delta_auprc,permutation_p,wilcoxon_p,n_drives,p_raw,p_holm,significant_0.05
0,lgbm vs rule,0.0954,0.0075,0.0156,7,0.0075,0.0374,True
1,lgbm vs logreg,0.0320,0.0574,0.0781,7,0.0574,0.1147,False
2,lgbm vs gru,0.0285,0.0274,0.0312,7,0.0274,0.0823,False
3,lgbm vs tcn,0.0135,0.0673,0.0156,7,0.0673,0.1147,False
4,lgbm vs transformer,0.0366,0.0075,0.0156,7,0.0075,0.0374,True


best model at the 2s horizon: lgbm


### 6.4 Cost of inference (RQ7)

A model that is 1% better and 10× slower is not better. Parameter count, model size and
per-sample latency, measured on this machine.

In [26]:
from hoproj.deploy.profile import model_footprint, profile_torch_model
from hoproj.models.registry import build_model, model_kind

period = float(BASE.get_path('data.target_period_s') or 1.0)
Lw = max(2, int(round(float(BASE.get_path('windows.length_s', 10.0)) / period)))
prof_rows = []
for name in [m for m in MODELS if model_kind(m) == 'torch']:
    cfg_m = load_config('base.yaml', adapter='curated_v1', model=name, overrides=BUDGET)
    mdl = build_model(name, cfg_m, A.n_features, len(horizons), d_cand=A.d_cand)
    for r in profile_torch_model(mdl.model, Lw, A.n_features,
                                 tuple(BUDGET['deploy']['profile_batch_sizes']),
                                 BUDGET['deploy']['profile_repeats']):
        prof_rows.append({'model': name, **r, **model_footprint(mdl.model)})
profile_tbl = pd.DataFrame(prof_rows)
display(profile_tbl[['model', 'batch_size', 'latency_ms_mean', 'per_sample_ms',
                     'predictions_per_second', 'n_parameters', 'state_size_mb']].round(4))

,model,batch_size,latency_ms_mean,per_sample_ms,predictions_per_second,n_parameters,state_size_mb
0,gru,1,1.4728,1.4728,679.0014,130500,0.4978
1,gru,32,2.4800,0.0775,12903.1857,130500,0.4978
2,gru,256,8.3833,0.0327,30537.0217,130500,0.4978
3,tcn,1,1.5334,1.5334,652.1306,101764,0.3882
4,tcn,32,3.5370,0.1105,9047.1644,101764,0.3882
5,tcn,256,13.9330,0.0544,18373.5961,101764,0.3882
6,transformer,1,1.3639,1.3639,733.1858,169572,0.8344
7,transformer,32,2.4953,0.0780,12824.3206,169572,0.8344
8,transformer,256,10.3049,0.0403,24842.5497,169572,0.8344


In [27]:
if len(profile_tbl):
    one = profile_tbl[profile_tbl['batch_size'] == 1]
    acc = main_results[main_results['horizon_s'] == horizons[j]].set_index('model')['auprc']
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    for _, r in one.iterrows():
        ax.scatter(r['latency_ms_mean'], acc.get(r['model'], np.nan),
                   s=40 + r['n_parameters'] / 3000, color=colors.get(r['model']))
        ax.annotate(r['model'], (r['latency_ms_mean'], acc.get(r['model'], np.nan)),
                    fontsize=8, xytext=(5, 3), textcoords='offset points')
    ax.set_xlabel('single-sample CPU latency (ms)')
    ax.set_ylabel(f'AUPRC @ {horizons[j]:g}s')
    ax.set_title('accuracy against inference cost (marker size = parameters)')
    plt.show()

---
## 7. R25 — the leakage experiment (RQ3)

The same models, the same features, the same horizons — evaluated three ways:

| condition | what it does | status |
|---|---|---|
| `random_row` | shuffles individual samples between train and test | **control only** |
| `grouped_drive` | whole drives assigned to one partition | the real protocol |
| `external_route` | an entire unseen corridor | section 10 |

Adjacent samples in a drive are almost identical. Random row splitting puts near-duplicates
on both sides of the wall; whatever that buys is not generalisation. Measuring the gap is the
point — it converts a methodological complaint into a number.

In [28]:
LEAK_MODELS = ['lgbm', 'gru']
leak_rows = []
for cond in ['random_row', 'grouped_drive']:
    A_c = assemble(feats, labels, BASE, splits[cond], PATHS['interim'] / 'wcache')
    for name in LEAK_MODELS:
        cfg_m = load_config('base.yaml', adapter='curated_v1', model=name, overrides=BUDGET)
        r = train_and_evaluate(cfg_m, feats, labels, ho, splits[cond], name,
                               feature_set='rf_mob_hist_qoe', assembled=A_c)
        leak_rows.append(r.horizon_table)
        print(f'{cond:<15} {name:<8} done')
leak = pd.concat(leak_rows, ignore_index=True)
infl = ST.leakage_inflation(leak, 'auprc')
display(infl.round(4))

16:13:53 | INFO    | hoproj.transforms      | dropping 9 degenerate features (train-only decision): ['mask_serving_rsrp', 'mask_serving_rsrq', 'mask_serving_sinr', 'mask_dl_tp_kbps', 'mask_rtt_ms', 'mask_pkt_loss_pct', 'mask_nbr1_rsrp', 'mask_nbr2_rsrp'] ...


16:13:53 | INFO    | hoproj.transforms      | fitted robust transform on 31760 train rows x 138 columns


16:13:54 | INFO    | hoproj.windows         | window index: 80412 windows of length 10 (stride 1) over 81 drives


16:13:54 | INFO    | hoproj.windows         | materialised windows: shape=(31537, 10, 138) (166.0 MB)


16:13:54 | INFO    | hoproj.assemble        | part=train  rows= 31760 windows= 31537 drives= 46


...

16:18:32 | INFO    | hoproj.wrapper         | gru ep005 train=0.8099 val=0.9140 metric=0.3281 (best 0.3394 @ep4)


16:18:56 | INFO    | hoproj.wrapper         | gru ep010 train=0.4536 val=1.6282 metric=0.2664 (best 0.3394 @ep4)


16:18:56 | INFO    | hoproj.wrapper         | gru early stop at epoch 10


grouped_drive   gru      done


condition,model,horizon_s,grouped_drive,random_row,delta_grouped,relative_inflation_grouped
0,gru,1.0,0.1869,0.2967,0.1097,0.5869
1,gru,2.0,0.2770,0.5790,0.3020,1.0902
2,gru,3.0,0.3554,0.6981,0.3427,0.9644
3,gru,5.0,0.4674,0.7882,0.3208,0.6864
4,lgbm,1.0,0.1910,0.1807,-0.0104,-0.0542
5,lgbm,2.0,0.3055,0.3407,0.0353,0.1154
6,lgbm,3.0,0.3870,0.4824,0.0955,0.2467
7,lgbm,5.0,0.4944,0.6392,0.1448,0.2929


In [29]:
if 'relative_inflation_grouped' in infl:
    fig, ax = plt.subplots(figsize=(7, 3.4))
    w = 0.35
    xs = np.arange(len(infl))
    ax.bar(xs - w / 2, infl['grouped_drive'], w, label='grouped by drive (honest)',
           color='#264653')
    ax.bar(xs + w / 2, infl['random_row'], w, label='random row split (inflated)',
           color='#e76f51')
    ax.set_xticks(xs)
    ax.set_xticklabels([f"{r['model']}\n{r['horizon_s']:g}s" for _, r in infl.iterrows()],
                       fontsize=7)
    ax.set_ylabel('AUPRC'); ax.set_title('What the evaluation protocol alone is worth')
    ax.legend(fontsize=8)
    plt.show()
    print('mean relative inflation from random row splitting: '
          f"{infl['relative_inflation_grouped'].mean():.1%}")

mean relative inflation from random row splitting: 49.1%


---
## 8. R13 — feature ablation (RQ5)

Does anything beyond the radio measurements earn its place? Four nested feature sets, and
separately the two geographic regimes: *topology-agnostic* (no GPS, no route, no cell ID)
against *context-rich*. A large gain from context is not good news — it means the model is
learning where it is rather than what the radio is doing.

In [30]:
FSETS = {'rf': ['rf'], 'rf+mob': ['rf', 'mobility'],
         'rf+mob+hist': ['rf', 'mobility', 'history'],
         'rf+mob+hist+qoe': ['rf', 'mobility', 'history', 'qoe']}
ab_rows = []
for fname, blocks_ in FSETS.items():
    cfg_f = Config(deep_merge(load_config('base.yaml', adapter='curated_v1', model='gru',
                                          overrides=BUDGET),
                              {'features': {'blocks': blocks_}}))
    A_f = assemble(feats, labels, cfg_f, splits['grouped_drive'], PATHS['interim'] / 'wcache')
    r = train_and_evaluate(cfg_f, feats, labels, ho, splits['grouped_drive'], 'gru',
                           feature_set=fname, assembled=A_f)
    t = r.horizon_table.copy(); t['feature_set'] = fname; t['n_inputs'] = A_f.n_features
    ab_rows.append(t)
    print(f'{fname:<18} {A_f.n_features:>4} inputs  done')
ablation = pd.concat(ab_rows, ignore_index=True)
display(ablation.pivot_table(index='feature_set', columns='horizon_s', values='auprc')
        .reindex(list(FSETS)).round(4))

16:19:01 | INFO    | hoproj.transforms      | dropping 6 degenerate features (train-only decision): ['mask_serving_rsrp', 'mask_serving_rsrq', 'mask_serving_sinr', 'mask_nbr1_rsrp', 'mask_nbr2_rsrp', 'mask_nbr3_rsrp']


16:19:01 | INFO    | hoproj.transforms      | fitted robust transform on 29027 train rows x 83 columns


16:19:02 | INFO    | hoproj.windows         | window index: 80412 windows of length 10 (stride 1) over 81 drives


16:19:02 | INFO    | hoproj.windows         | materialised windows: shape=(28802, 10, 83) (91.2 MB)


16:19:02 | INFO    | hoproj.assemble        | part=train  rows= 29027 windows= 28802 drives= 25


...

16:22:26 | INFO    | hoproj.wrapper         | gru ep005 train=0.8099 val=0.9140 metric=0.3281 (best 0.3394 @ep4)


16:22:53 | INFO    | hoproj.wrapper         | gru ep010 train=0.4536 val=1.6282 metric=0.2664 (best 0.3394 @ep4)


16:22:53 | INFO    | hoproj.wrapper         | gru early stop at epoch 10


rf+mob+hist+qoe     138 inputs  done


horizon_s,1.0,2.0,3.0,5.0
feature_set,,,,
rf,0.1740,0.2605,0.3432,0.4606
rf+mob,0.1797,0.2615,0.3380,0.4339
rf+mob+hist,0.1910,0.2836,0.3639,0.4795
rf+mob+hist+qoe,0.1869,0.2770,0.3554,0.4674


In [31]:
reg_rows = []
for regime in ['topology_agnostic', 'context_rich']:
    cfg_r = Config(deep_merge(load_config('base.yaml', adapter='curated_v1', model='gru',
                                          overrides=BUDGET),
                              {'features': {'regime': regime}}))
    A_r = assemble(feats, labels, cfg_r, splits['grouped_drive'], PATHS['interim'] / 'wcache')
    r = train_and_evaluate(cfg_r, feats, labels, ho, splits['grouped_drive'], 'gru',
                           feature_set='rf_mob_hist_qoe', assembled=A_r)
    t = r.horizon_table.copy(); t['regime'] = regime
    reg_rows.append(t)
    print(f'{regime:<20} {A_r.n_features:>4} inputs  done')
regimes = pd.concat(reg_rows, ignore_index=True)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 3.6))
for fname in FSETS:
    g = ablation[ablation['feature_set'] == fname].sort_values('horizon_s')
    a1.plot(g['horizon_s'], g['auprc'], marker='o', label=fname)
for reg, g in regimes.groupby('regime'):
    g = g.sort_values('horizon_s')
    a2.plot(g['horizon_s'], g['auprc'], marker='s', label=reg)
a1.set_xlabel('horizon (s)'); a1.set_ylabel('AUPRC'); a1.set_title('feature blocks')
a1.legend(fontsize=7)
a2.set_xlabel('horizon (s)'); a2.set_ylabel('AUPRC')
a2.set_title('topology-agnostic vs context-rich'); a2.legend(fontsize=7)
plt.show()

16:22:59 | INFO    | hoproj.transforms      | dropping 9 degenerate features (train-only decision): ['mask_serving_rsrp', 'mask_serving_rsrq', 'mask_serving_sinr', 'mask_dl_tp_kbps', 'mask_rtt_ms', 'mask_pkt_loss_pct', 'mask_nbr1_rsrp', 'mask_nbr2_rsrp'] ...


16:23:00 | INFO    | hoproj.transforms      | fitted robust transform on 29027 train rows x 138 columns


16:23:00 | INFO    | hoproj.windows         | window index: 80412 windows of length 10 (stride 1) over 81 drives


16:23:00 | INFO    | hoproj.windows         | materialised windows: shape=(28802, 10, 138) (151.6 MB)


16:23:00 | INFO    | hoproj.assemble        | part=train  rows= 29027 windows= 28802 drives= 25


...

16:24:04 | INFO    | hoproj.wrapper         | gru ep001 train=1.1316 val=1.0201 metric=0.2587 (best 0.2587 @ep1)


16:24:24 | INFO    | hoproj.wrapper         | gru ep005 train=0.7741 val=0.9297 metric=0.3303 (best 0.3396 @ep4)


16:24:49 | INFO    | hoproj.wrapper         | gru ep010 train=0.4250 val=1.8332 metric=0.2883 (best 0.3396 @ep4)


16:24:49 | INFO    | hoproj.wrapper         | gru early stop at epoch 10


context_rich          153 inputs  done


---
## 9. R14 — uncertainty, OOD and abstention (RQ4)

A model that is accurate on average can still be confidently wrong on unfamiliar ground, and
that is the failure mode that matters for mobility control. Four steps, in this order:

1. a **deep ensemble** — identical models, different seeds; their disagreement is the
   epistemic-uncertainty signal;
2. **temperature scaling** fitted on the calibration drives, which the models never saw and
   which never influenced early stopping;
3. **split conformal** over whole calibration drives, giving prediction sets with a stated
   coverage target — and reporting the coverage actually achieved, plus its spread across
   drives, rather than trusting the exchangeability assumption;
4. an **abstention policy** — reject the most uncertain samples and see whether the errors
   that remain are fewer.

In [32]:
from hoproj.uncertainty.calibration import (TemperatureScaler, expected_calibration_error,
                                            reliability_curve)
from hoproj.uncertainty.conformal import BinaryMondrianConformal
from hoproj.uncertainty import abstention as AB, ood as OOD
from hoproj.pipeline.stage03_uncertainty import train_ensemble

cfg_u = load_config('base.yaml', adapter='curated_v1', model='gru', overrides=BUDGET)
ens, ds = train_ensemble(cfg_u, A, 'gru', int(BUDGET['uncertainty']['ensemble_size']))

pred = {p: ens.predict(ds[p]) for p in ('train', 'val', 'calib', 'test') if len(ds[p])}
pos = {p: A.windows[p].end_pos for p in pred}
Y = {p: A.Y_ho[pos[p]] for p in pred}
M = {p: A.M_ho[pos[p]] for p in pred}
meta = {p: window_meta(A, p) for p in pred}
print('ensemble of', ens.size, 'GRUs')

16:24:55 | INFO    | hoproj                 | START ensemble member 1/3 (seed 1337)


16:24:55 | INFO    | hoproj.wrapper         | gru pos_weight per horizon: [16.209999084472656, 8.369999885559082, 5.739999771118164, 3.619999885559082]


16:25:01 | INFO    | hoproj.wrapper         | gru ep001 train=1.1246 val=1.0131 metric=0.2619 (best 0.2619 @ep1)


16:25:24 | INFO    | hoproj.wrapper         | gru ep005 train=0.8099 val=0.9140 metric=0.3281 (best 0.3394 @ep4)


16:25:53 | INFO    | hoproj.wrapper         | gru ep010 train=0.4536 val=1.6282 metric=0.2664 (best 0.3394 @ep4)


...

16:27:45 | INFO    | hoproj.wrapper         | gru ep010 train=0.4469 val=1.8114 metric=0.2861 (best 0.3518 @ep5)


16:27:50 | INFO    | hoproj.wrapper         | gru ep011 train=0.3908 val=1.9252 metric=0.2910 (best 0.3518 @ep5)


16:27:50 | INFO    | hoproj.wrapper         | gru early stop at epoch 11


16:27:50 | INFO    | hoproj                 | DONE  ensemble member 3/3 (seed 1539) (59.64s)


ensemble of 3 GRUs


In [33]:
scaler = TemperatureScaler().fit(pred['calib']['mean'], Y['calib'], M['calib'])
cal = {p: scaler.transform(v['mean']) for p, v in pred.items()}

rows = []
for jj, h in enumerate(horizons):
    s = M['test'][:, jj].astype(bool)
    rows.append({'horizon_s': h,
                 'ECE_raw': expected_calibration_error(Y['test'][s, jj], pred['test']['mean'][s, jj]),
                 'ECE_calibrated': expected_calibration_error(Y['test'][s, jj], cal['test'][s, jj]),
                 'temperature': scaler.T_[jj]})
display(pd.DataFrame(rows).round(4))

16:27:57 | INFO    | hoproj.calibration     | fitted temperatures: [1.203, 1.256, 1.279, 1.227]


,horizon_s,ECE_raw,ECE_calibrated,temperature
0,1.0,0.2395,0.2478,1.2029
1,2.0,0.2109,0.2194,1.2561
2,3.0,0.1842,0.1921,1.2787
3,5.0,0.1466,0.1522,1.2273


In [34]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.5, 4.2))
for ax, probs, title in ((a1, pred['test']['mean'], 'before calibration'),
                         (a2, cal['test'], 'after temperature scaling')):
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.6)
    for jj, h in enumerate(horizons):
        s = M['test'][:, jj].astype(bool)
        c, a_, _ = reliability_curve(Y['test'][s, jj], probs[s, jj])
        ax.plot(c, a_, marker='o', ms=3, label=f'{h:g}s')
    ax.set_xlabel('predicted probability'); ax.set_ylabel('observed frequency')
    ax.set_title(title); ax.legend(fontsize=7)
plt.show()

In [35]:
cp = BinaryMondrianConformal(float(BASE.get_path('uncertainty.conformal.alpha', 0.1)))
cp.fit(cal['calib'], Y['calib'], M['calib'], meta['calib']['drive_id'].to_numpy())
cov = cp.coverage(cal['test'], Y['test'], M['test'], meta['test']['drive_id'].to_numpy())
cov_tbl = pd.DataFrame([{'horizon_s': horizons[k], **v} for k, v in cov.items()])
display(cov_tbl.round(4))
print(f'nominal coverage target: {1 - cp.alpha:.0%}')

16:27:57 | INFO    | hoproj.conformal       | conformal thresholds (alpha=0.10): [[0.678, 0.69], [0.681, 0.687], [0.682, 0.691], [0.691, 0.698]]


,horizon_s,empirical_coverage,per_drive_coverage_min,per_drive_coverage_median,per_drive_coverage_iqr,n_drives,mean_set_size,singleton_rate,ambiguous_rate,empty_rate,n
0,1.0,0.8955,0.8697,0.8943,0.0261,7,1.3297,0.6703,0.3297,0.0,8347
1,2.0,0.8924,0.8696,0.8846,0.0259,7,1.3426,0.6574,0.3426,0.0,8340
2,3.0,0.8908,0.8715,0.8867,0.0221,7,1.3534,0.6466,0.3534,0.0,8333
3,5.0,0.8929,0.8673,0.8901,0.0108,7,1.3796,0.6204,0.3796,0.0,8321


nominal coverage target: 90%


### 9.1 Does abstention buy anything?

The curve below rejects samples in order of ensemble disagreement. `fn_risk` is the fraction
of real upcoming handovers that are missed among the samples the model still answers — the
error a mobility controller actually pays for. If the curve falls as coverage drops, the
uncertainty signal is carrying real information about where the model is wrong.

In [36]:
scorer = OOD.DisagreementScorer()
score = {p: scorer.score(pred[p]['std']) for p in pred}

covs = [float(c) for c in BASE.get_path('uncertainty.abstention.target_coverages')]
pool = np.concatenate([score[p] for p in ('val', 'calib') if p in score])
taus = {c: AB.select_threshold(pool, c) for c in covs}

from hoproj.eval.metrics import recall_at_fpr
sel_v = M['val'][:, j].astype(bool)
_, thr_op = recall_at_fpr(Y['val'][sel_v, j], cal['val'][sel_v, j], 0.05)
thr_op = float(thr_op) if np.isfinite(thr_op) else 0.5

sel_t = M['test'][:, j].astype(bool)
rc = AB.risk_coverage_curve(Y['test'][sel_t, j], cal['test'][sel_t, j], score['test'][sel_t],
                            coverages=covs, taus=taus, decision_threshold=thr_op)
display(rc.round(4))

,target_coverage,tau,realised_coverage,n_accepted,risk,fn_risk,fp_rate,positive_rate_accepted,auprc
0,1.00,0.3012,1.0000,8340,0.1290,0.7845,0.0520,0.1052,0.2853
1,0.95,0.1385,0.9548,7963,0.1300,0.7739,0.0544,0.1050,0.2895
2,0.90,0.1108,0.9054,7551,0.1300,0.7586,0.0573,0.1037,0.2960
3,0.80,0.0809,0.8054,6717,0.1337,0.7296,0.0645,0.1041,0.3076
4,0.70,0.0621,0.6972,5815,0.1360,0.6840,0.0736,0.1023,0.3193
5,0.60,0.0481,0.5988,4994,0.1328,0.6105,0.0825,0.0951,0.3329
6,0.50,0.0370,0.5080,4237,0.1286,0.5302,0.0909,0.0859,0.3455


In [37]:
fig, ax = plt.subplots(figsize=(6.6, 3.6))
d = rc.sort_values('realised_coverage')
ax.plot(d['realised_coverage'], d['fn_risk'], marker='o', color='#e76f51',
        label='missed handovers among answered')
ax.plot(d['realised_coverage'], d['risk'], marker='s', color='#264653',
        label='overall error rate among answered')
ax.set_xlabel('coverage — fraction of samples the model answers')
ax.set_ylabel('risk'); ax.set_title(f'risk against coverage, {horizons[j]:g}s horizon')
ax.legend(fontsize=8)
plt.show()

---
## 10. R15 — the locked external route (RQ2, RQ6)

Everything above used two corridors. A third was set aside before any model was built and
has contributed nothing: not a scaling statistic, not a hyper-parameter, not a threshold.

The pipeline enforces that with a **freeze manifest**. Writing it records the config
fingerprint, the chosen model, the calibrator and the abstention thresholds. Stage 04 refuses
to run without one, and if the fingerprint later changes it says so in the report. That is
the only way the claim "we did not peek" is auditable rather than asserted.

In [38]:
from hoproj.pipeline import stage04_external as S4

manifest_path = S4.write_freeze_manifest(BASE, PATHS, 'gru',
                                          notes='frozen from the notebook run')
print(json.dumps(json.loads(Path(manifest_path).read_text())
                 if str(manifest_path).endswith('.json') else {}, indent=2)[:900])

16:27:58 | INFO    | hoproj.stage04         | wrote freeze manifest to /home/claude/ho-pipeline/artifacts/freeze_manifest.json


{
  "config_fingerprint": "ead9c9d0bfad",
  "frozen_model": "gru",
  "external_route": "kuril_badda_rampura_malibagh",
  "horizons_s": [
    1.0,
    2.0,
    3.0,
    5.0
  ],
  "window_length_s": 10.0,
  "feature_regime": "topology_agnostic",
  "feature_blocks": [
    "rf",
    "mobility",
    "history",
    "qoe"
  ],
  "uncertainty_policy": null,
  "notes": "frozen from the notebook run"
}


In [39]:
ext_out = S4.run(BASE, PATHS, 'gru', require_freeze=True)
ext_res = ext_out['results']
display(ext_res[[c for c in ['horizon_s', 'positive_rate', 'auprc', 'auprc_ci_low',
                             'auprc_ci_high', 'auroc', 'recall_at_fpr0.05', 'ece']
                 if c in ext_res]].round(4))
display(ext_out['events'][[c for c in ['horizon_s', 'n_events', 'event_detection_rate',
                                       'median_lead_time_s', 'false_alarms_per_hour']
                           if c in ext_out['events']]].round(3))

16:28:00 | INFO    | hoproj.transforms      | dropping 9 degenerate features (train-only decision): ['mask_serving_rsrp', 'mask_serving_rsrq', 'mask_serving_sinr', 'mask_dl_tp_kbps', 'mask_rtt_ms', 'mask_pkt_loss_pct', 'mask_nbr1_rsrp', 'mask_nbr2_rsrp'] ...


16:28:00 | INFO    | hoproj.transforms      | fitted robust transform on 29027 train rows x 138 columns


16:28:01 | INFO    | hoproj.windows         | window index: 80412 windows of length 10 (stride 1) over 81 drives


16:28:01 | INFO    | hoproj.windows         | materialised windows: shape=(28802, 10, 138) (151.6 MB)


16:28:01 | INFO    | hoproj.assemble        | part=train  rows= 29027 windows= 28802 drives= 25


...

16:30:45 | INFO    | hoproj.calibration     | fitted temperatures: [1.203, 1.256, 1.279, 1.227]


16:30:54 | INFO    | hoproj.report          | wrote report /home/claude/ho-pipeline/reports/04_external_route.md


16:30:54 | INFO    | hoproj.stage04         | stage 04 complete


,horizon_s,positive_rate,auprc,auprc_ci_low,auprc_ci_high,auroc,recall_at_fpr0.05,ece
0,1.0,0.0703,0.2703,0.2487,0.2956,0.8336,0.3109,0.2789
1,2.0,0.1258,0.3874,0.3539,0.4155,0.8243,0.3011,0.2420
2,3.0,0.1711,0.4618,0.4263,0.4928,0.8162,0.2840,0.2171
3,5.0,0.2445,0.5660,0.5302,0.5989,0.8138,0.2914,0.1700


,horizon_s,n_events,event_detection_rate,median_lead_time_s,false_alarms_per_hour
0,1.0,2341,0.143,1.0,56.250
1,2.0,2341,0.220,2.0,51.153
2,3.0,2341,0.299,2.0,44.328
3,5.0,2341,0.374,3.0,30.268


### 10.1 The number that decides the paper

Development performance against locked-route performance. A small gap means the model learned
radio dynamics. A large gap means it learned two particular corridors, and the honest thing
is to report that.

In [40]:
dev = main_results[main_results['model'] == 'gru'].set_index('horizon_s')['auprc']
ext = ext_res.set_index('horizon_s')['auprc']
gap = pd.DataFrame({'development (grouped drives)': dev, 'locked external route': ext})
gap['drop'] = gap.iloc[:, 0] - gap.iloc[:, 1]
gap['relative_drop'] = gap['drop'] / gap.iloc[:, 0]
display(gap.round(4))

fig, ax = plt.subplots(figsize=(6.6, 3.5))
xs = np.arange(len(gap)); w = 0.38
ax.bar(xs - w / 2, gap.iloc[:, 0], w, label='development', color='#264653')
ax.bar(xs + w / 2, gap.iloc[:, 1], w, label='locked external route', color='#e9c46a')
ax.set_xticks(xs); ax.set_xticklabels([f'{h:g}s' for h in gap.index])
ax.set_ylabel('AUPRC'); ax.set_title('Does it survive an unseen corridor?')
ax.legend(fontsize=8)
plt.show()

,development (grouped drives),locked external route,drop,relative_drop
horizon_s,,,,
1.0,0.1869,0.2703,-0.0833,-0.4458
2.0,0.2770,0.3874,-0.1104,-0.3984
3.0,0.3554,0.4618,-0.1064,-0.2993
5.0,0.4674,0.5660,-0.0986,-0.2109


In [41]:
summary_ext = json.loads((PATHS['artifacts'] / 'external_summary.json').read_text())
print('geographic OOD separation (dev vs external):')
for k, v in summary_ext['ood_geographic'].items():
    print(f'  {k:<22} {v}')
print()
print('how many external cells were never seen in development:')
ext_route = BASE.get_path('splits.external_route')
dev_mask = (feats['route_id'] != ext_route).to_numpy()
ext_mask = (feats['route_id'] == ext_route).to_numpy()
ov = cell_overlap(samples, dev_mask, ext_mask)
for k, v in ov.items():
    if k != 'unseen_examples':
        print(f'  {k:<26} {v}')

geographic OOD separation (dev vs external):
  auroc                  0.5422845038169253
  detection_rate         0.05792141402763939
  threshold              0.13852091394364818
  fpr_in_distribution    0.050031699070160605
  n_in                   18928
  n_out                  23083

how many external cells were never seen in development:
  dev_cells                  43
  external_cells             41
  unseen_external_cells      3
  unseen_fraction            0.07317073170731707


In [42]:
ext_rc = ext_out['risk_coverage']
display(ext_rc.round(4))
fig, ax = plt.subplots(figsize=(6.6, 3.5))
d1 = rc.sort_values('realised_coverage'); d2 = ext_rc.sort_values('realised_coverage')
ax.plot(d1['realised_coverage'], d1['fn_risk'], marker='o', color='#264653',
        label='development drives')
ax.plot(d2['realised_coverage'], d2['fn_risk'], marker='s', color='#e76f51',
        label='locked external route')
ax.set_xlabel('coverage'); ax.set_ylabel('missed handovers among answered')
ax.set_title('abstention with thresholds frozen before the route was opened')
ax.legend(fontsize=8)
plt.show()

,target_coverage,tau,realised_coverage,n_accepted,risk,fn_risk,fp_rate,positive_rate_accepted,auprc,part,horizon_s
0,0.50,0.0370,0.4070,7735,0.2293,0.0950,0.2515,0.1416,0.4809,external_route,2.0
1,0.60,0.0481,0.5220,9920,0.2634,0.1202,0.2875,0.1443,0.4549,external_route,2.0
2,0.70,0.0621,0.6458,12273,0.2785,0.1451,0.3005,0.1415,0.4342,external_route,2.0
3,0.80,0.0809,0.7706,14644,0.2887,0.1670,0.3082,0.1378,0.4140,external_route,2.0
4,0.90,0.1108,0.8827,16774,0.2904,0.1912,0.3057,0.1335,0.3998,external_route,2.0
5,0.95,0.1385,0.9334,17738,0.2898,0.2013,0.3031,0.1311,0.3938,external_route,2.0
6,1.00,0.3012,0.9999,19003,0.2815,0.2179,0.2906,0.1258,0.3874,external_route,2.0


---
## 11. R16 — consolidated report and artefacts

In [43]:
from hoproj.eval import report as RPT
from hoproj.pipeline import stage05_report as S5

RPT.save_table(main_results, PATHS['reports'], 'main_results')
if len(main_events):
    RPT.save_table(main_events, PATHS['reports'], 'main_events')
RPT.save_table(leak, PATHS['reports'], 'leakage_study_results')
RPT.save_table(infl, PATHS['reports'], 'leakage_study_leakage_inflation')
RPT.save_table(ablation, PATHS['reports'], 'ablation_features_results')
RPT.save_table(regimes, PATHS['reports'], 'ablation_regimes')
RPT.save_table(rc, PATHS['reports'], 'risk_coverage')
RPT.save_table(cov_tbl, PATHS['reports'], 'conformal_coverage')
if len(profile_tbl):
    RPT.save_table(profile_tbl, PATHS['reports'], 'deployment_profile')

out = S5.run(BASE, PATHS)
print('consolidated report:', out)

16:30:54 | INFO    | hoproj.report          | wrote report /home/claude/ho-pipeline/reports/00_thesis_report.md


consolidated report: /home/claude/ho-pipeline/reports/00_thesis_report.md


In [44]:
print('generated tables:')
for p in sorted((PATHS['reports'] / 'tables').glob('*.csv')):
    print('  ', p.name)
print()
print('generated figures:')
for p in sorted((PATHS['reports'] / 'figures').glob('*.png')):
    print('  ', p.name)

generated tables:
   ablation_features_results.csv
   ablation_regimes.csv
   authenticity_audit.csv
   authenticity_audit_fixed.csv
   conformal_coverage.csv
   deployment_profile.csv
   external_route_events.csv
   external_route_results.csv
   external_route_risk_coverage.csv
   leakage_study_leakage_inflation.csv
   leakage_study_results.csv
   main_events.csv
   main_results.csv
   risk_coverage.csv

generated figures:
   external_reliability.png
   external_risk_coverage.png


---
## What to take away

Fill these in from the cells above when writing up:

- **RQ1** — sequence models against the LightGBM snapshot baseline: compare section 6.1
  (AUPRC, which may be close) with 6.2 (false alarms per hour, where the gap is usually
  much larger). The operating-point difference is the more publishable finding.
- **RQ2** — the development-to-external drop in section 10.1.
- **RQ3** — the inflation from random row splitting in section 7.
- **RQ4** — whether the risk–coverage curve in 9.1 falls, and whether conformal coverage in
  section 9 lands on its nominal target.
- **RQ5** — whether mobility, history and QoE blocks add anything in section 8, and whether
  context-rich beats topology-agnostic by enough to worry about memorisation.
- **RQ7** — the accuracy-versus-latency scatter in 6.4.

Negative results stay in. A Transformer that does not beat a GRU, or a QoE block that adds
nothing, is a finding — and a paper that reports one is harder to argue with than a paper
that does not.

---
# Part II: Modern XCAL Pipeline — Decoded Signalling, Survival Hazard & Conformal Risk (Stages 12–21)

This second part implements the final thesis evidence base:
- **4 drive-test campaigns** (10, 12, 13, and 15 Sept highway corridor): 57 drives, 10,260 s, 938 handovers across 6 carrier frequencies.
- **Dynamic ASN.1 RRC timeline reconstruction** fixing stateful `measId`/`reportConfigId` misattribution affecting 99.4% of frames.
- **Grouped whole-drive K-fold holdout** eliminating temporal autocorrelation leakage.
- **Discrete-time survival hazard formulation** ($S(t) = \prod_{k \le t}(1 - h(k))$) guaranteeing 0% monotonicity violations.
- **Conformal Risk Control (CRC)** establishing distribution-free miss-rate guarantees ($\mathbb{E}[L] \le lpha$).
- **Shafi et al. departmental baseline re-implementation** under identical drive-test conditions.
- **Leave-One-Capture-Out (LOCO) high-speed highway transfer** and Astana public dataset validation.


In [ ]:
# Stage 12: Materialise Pooled XCAL Signalling Dataset
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
os.environ['HOPROJ_ROOT'] = str(ROOT)

from hoproj.pipeline.stage12_xcal_prepare import load_xcal_tables, CAPTURES, HORIZONS

print(f"Configured XCAL Captures: {len(CAPTURES)}")
for tag, raw_dir, csv_name, sig_name in CAPTURES:
    print(f"  [{tag}] {raw_dir} -> {csv_name} + {sig_name}")

# Inspect pooled dataset summary
xcal_summary_path = ROOT / 'reports_xcal' / 'tables' / 'xcal_capture_summary.csv'
if xcal_summary_path.exists():
    df_summary = pd.read_csv(xcal_summary_path)
    display(df_summary)


### 11.1 Grouped K-Fold Benchmark Across 57 Drives (Stage 13)
Evaluates LightGBM, Logistic Regression, MLP, and GRU under grouped drive-level splits.
Every drive is tested exactly once, with all scalers and thresholds fitted strictly inside the fold.


In [ ]:
# Stage 13: Primary Empirical Benchmark
main_res_path = ROOT / 'reports_xcal' / 'tables' / 'xcal_main.csv'
if main_res_path.exists():
    df_main = pd.read_csv(main_res_path)
    # Filter to main 1 s and 5 s horizons
    display(df_main[df_main['horizon_s'].isin([1.0, 5.0])][
        ['model', 'feature_set', 'horizon_s', 'auroc_mean', 'auprc_mean', 'auprc_lift_mean', 'ece_mean']
    ].sort_values(['horizon_s', 'auroc_mean'], ascending=[True, False]))


### 11.2 Coherent Discrete-Time Survival Hazard Analysis (Stage 14)
Evaluates the product-limit discrete hazard formulation ($S(t) = \prod_{k \le t}(1 - h(k))$) against unconstrained multi-head binary classifiers.
Enforces 0% monotonicity violations ($P(T \le 1) \le P(T \le 2) \le \dots$) with zero mathematical overhead.


In [ ]:
# Stage 14: Hazard Coherence & Calibration Evaluation
hazard_summary_path = ROOT / 'reports_xcal' / 'tables' / 'hazard_fair_summary.csv'
monotone_path = ROOT / 'reports_xcal' / 'tables' / 'horizon_monotonicity_fair.csv'

if hazard_summary_path.exists():
    print("=== Hazard Model vs Independent Heads Performance ===")
    display(pd.read_csv(hazard_summary_path))

if monotone_path.exists():
    print("=== Monotonicity Violation Rates across Test Drives ===")
    display(pd.read_csv(monotone_path))


### 11.3 Conformal Risk Control & Hawkes Point Process (Stage 15)
Computes distribution-free finite-sample guarantees on missed handovers via Conformal Risk Control (CRC).
Fits a Hawkes self-exciting point process showing that 60% of handovers arrive as secondary aftershocks ($n=0.611$).


In [ ]:
# Stage 15: Conformal Risk Control Frontier & Hawkes Fit
rc_path = ROOT / 'reports_xcal' / 'tables' / 'risk_control_frontier.csv'
hawkes_path = ROOT / 'reports_xcal' / 'tables' / 'hawkes_goodness_of_fit.csv'

if rc_path.exists():
    print("=== Conformal Risk Frontier (Alpha vs Empirical Loss vs Alarm Budget) ===")
    display(pd.read_csv(rc_path).head(10))

if hawkes_path.exists():
    print("=== Hawkes Goodness-of-Fit & Branching Ratio ===")
    display(pd.read_csv(hawkes_path))


### 11.4 Feature Representation & Signalling Ablation (Stage 16)
Quantifies predictive capacity across feature representations (Full vs. RF+Mobility+History vs. Pure Signalling)
and evaluates ping-pong rate sensitivity across standard 3GPP and literature definitions.


In [ ]:
# Stage 16: Feature Ablation & Ping-Pong Analysis
sig_abl_path = ROOT / 'reports_xcal' / 'tables' / 'signalling_ablation.csv'
pp_def_path = ROOT / 'reports_xcal' / 'tables' / 'pingpong_definitions.csv'

if sig_abl_path.exists():
    print("=== Feature Representation Ablation across Horizons ===")
    display(pd.read_csv(sig_abl_path))

if pp_def_path.exists():
    print("=== Ping-Pong Definition Sensitivity on the same 938 Handovers ===")
    display(pd.read_csv(pp_def_path))


### 11.5 Equal 50-Trial Bayesian Tuning Budget (Stage 18)
Eliminates strawman baseline bias by providing identical Optuna Bayesian hyperparameter search budgets to all model families.


In [ ]:
# Stage 18: Equal Tuning Budget Analysis
tuning_path = ROOT / 'reports_xcal' / 'tables' / 'tuning_budget_all.csv'
if tuning_path.exists():
    display(pd.read_csv(tuning_path))


### 11.6 Departmental Baseline Comparison: Shafi et al. Re-Implementation (Stage 20)
Direct head-to-head re-implementation of Shafi et al.'s Q-learning reinforcement learning baseline on our drive-test dataset.


In [ ]:
# Stage 20: Departmental Baseline Head-to-Head
dept_path = ROOT / 'reports_xcal' / 'tables' / 'departmental_baseline_summary.csv'
if dept_path.exists():
    display(pd.read_csv(dept_path))


### 11.7 Leave-One-Capture-Out Highway Transfer (Stage 21)
Evaluates spatial and velocity transfer: trained on 25 km/h urban drives and tested zero-shot on 60 km/h highway corridor (Capture 4).


In [ ]:
# Stage 21: Leave-One-Capture-Out Transfer
cap_trans_path = ROOT / 'reports_xcal' / 'tables' / 'capture_transfer.csv'
if cap_trans_path.exists():
    display(pd.read_csv(cap_trans_path))


---
## 12. Audited Revision Pipeline (Stages 22–31)

This section incorporates the post-audit revision pipeline establishing:
- **Stage 22:** Timestamp alignment audit, evaluation protocol ladder (LOCO), discrete-time hazard coherence, conformal risk control (CRC), and audited lag-1 feature ablation.
- **Stage 23:** Signalling revision: 1-to-1 matching of Event A3 measurement reports to handover commands, non-deterministic conversion (34.6%), and completion dynamics.
- **Stage 24:** External public dataset domain transfer from pure L3 signaling.
- **Stage 26:** Temporal dynamics: feature block counts, ping-pong ladder, and Hawkes self-exciting point process ($n \approx 0.61$).
- **Stage 27:** External alignment control on Irish MMSys 2020 (instantaneous G-NetTrack logging).
- **Stage 28:** External replication of the row-time alignment artifact on US Tier-1 commercial networks (NUWiNS PAM 2025 AT&T & T-Mobile) and sub-second 100 ms native RRC auditing.
- **Stage 29:** Cross-instrument alignment mechanism and intra-second command geometry.
- **Stage 30:** Post-hoc window sensitivity analysis (purge, blanking, burst cutoffs).
- **Stage 31:** Unsifted real-world case studies (True Positives, False Positives, False Negatives).


In [ ]:
# Stage 22-31 Revision Runner and Status Inspection
import hoproj.pipeline.run_revision as rr
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'reports_rev').exists() else Path.cwd().parent
TAB_REV = ROOT / 'reports_rev' / 'tables'
print("=== Audited Revision Pipeline (Stages 22-31) ===")
print(f"Revision tables directory: {TAB_REV}")
print(f"Available revision tables: {len(list(TAB_REV.glob('*.csv')))}")

# Inspect key revision summary tables
summary_tables = {
    'c9_row_alignment_audit': 'Timestamp Alignment Audit (Grameenphone)',
    'c9ext_nuwins_alignment': 'External US Replication (NUWiNS AT&T / T-Mobile)',
    'c9ext_nuwins_subsecond': 'Sub-Second 100 ms Native RRC Auditing',
    'c2_protocol_ladder': 'Protocol Ladder & Lag-0 vs Lag-1 Inversion',
    'c3_main_loco': 'Main LOCO Multi-Horizon Benchmark',
    'c11_coherence_violations': 'Hazard Monotonicity Coherence',
    'c10_crc_frontier': 'Conformal Risk Control Frontier',
    'c18_ablation': 'Audited Feature Ablation (Lag-1)'
}

for tab, desc in summary_tables.items():
    p = TAB_REV / f'{tab}.csv'
    if p.exists():
        df = pd.read_csv(p)
        print(f"\n--- {desc} ({tab}.csv, {len(df)} rows) ---")
        display(df.head(3))


### 12.1 Execution Instructions for the Revision Pipeline
To re-run any specific revision stage or the entire revision suite:
- `python -m hoproj.pipeline.run_revision` (runs all stages 22 to 31)
- `python -m hoproj.pipeline.run_revision --only 22 26 30` (runs specific stages)
- `python -m hoproj.pipeline.run_revision --offline` (skips network-bound stage 28)

All generated tables are stored in `reports_rev/tables/` and publication figures in `reports_rev/figures/`.
